In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [2]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [3]:
categorical = list(df.dtypes[df.dtypes == 'object'].index)
numerical = [c for c in df.columns if c not in categorical and c != 'converted']

df.isnull().sum()   # check where values are missing

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

In [4]:
df['industry'].mode()
# or: df['industry'].value_counts()

0    technology
Name: industry, dtype: object

In [5]:
df[numerical].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [6]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train.converted.values
y_val = df_val.converted.values

del df_train['converted']
del df_val['converted']

In [7]:
for col in ['industry', 'location', 'lead_source', 'employment_status']:
    print(col, round(mutual_info_score(df_train[col], y_train), 2))

industry 0.0
location 0.0
lead_source 0.03
employment_status 0.02


In [8]:
features = categorical + numerical

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
X_val = dv.transform(df_val[features].to_dict(orient='records'))

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
base_acc = accuracy_score(y_val, y_pred)
print(round(base_acc, 2))

0.65


In [9]:
def train_and_score(feats, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_tr = dv.fit_transform(df_train[feats].to_dict(orient='records'))
    X_v = dv.transform(df_val[feats].to_dict(orient='records'))
    m = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    m.fit(X_tr, y_train)
    return accuracy_score(y_val, m.predict(X_v))

for f in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    feats = [x for x in features if x != f]
    acc = train_and_score(feats)
    print(f, base_acc - acc)

lead_source 0.0030000000000000027
number_of_courses_viewed 0.0020000000000000018
interaction_count 0.04400000000000004


In [10]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    print(C, round(train_and_score(features, C=C), 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
